# LAB02 — Dataset público: auditoría y SVM reproducible

**INF-8239 Ciencia de Datos II · Unidad 01 · Ejercicio 01**

**Estado: PENDIENTE DE APROBACIÓN DEL DOCENTE**

Este notebook sigue el orden requerido:

1. Objetivo · 2. Pregunta · 3. Dataset · 4. Fuente · 5. Licencia · 6. Descarga · 7. Carga · 8. Estructura · 9. Auditoría · 10. Diccionario · 11. Target · 12. Leakage · 13. Train/test · 14. Preprocessing · 15. Baseline · 16. SVM · 17. Métricas · 18. Análisis de errores · 19. Limitaciones · 20. Conclusión

## 1. Objetivo

Integrar las evidencias de LAB00, LAB01 y LAB02 en una entrega individual, ejecutable y reproducible: selección de dataset público, auditoría, prevención de data leakage, baseline, SVM reproducible, métricas y análisis de errores.

## 2. Pregunta de investigación

¿Es posible predecir, a partir de variables disponibles **antes** del resultado, si un cliente de una entidad bancaria portuguesa suscribirá un depósito a plazo (variable `y`) durante una campaña de telemarketing?

Se excluye explícitamente `duration` (duración de la última llamada) porque, según la documentación oficial de UCI, solo se conoce **después** de realizar la llamada y afecta fuertemente al resultado: incluirla sería data leakage.

## 3. Dataset y 4. Fuente

- **Nombre:** Bank Marketing
- **Fuente:** UCI Machine Learning Repository (dataset 222)
- **URL:** https://archive.ics.uci.edu/dataset/222/bank+marketing
- **DOI:** 10.24432/C5K306
- **Autores:** Sérgio Moro, Paulo Cortez, Rita (Universidad del Minho, Portugal)
- **Artículo:** *A data-driven approach to predict the success of bank telemarketing*, Decision Support Systems, 2014

**Candidato comparado:** Wine Quality (UCI 186, DOI 10.24432/C56S3T, CC BY 4.0). La comparación completa está en `docs/candidatos_dataset.md`.

## 5. Licencia

**Creative Commons Attribution 4.0 International (CC BY 4.0)** — verificada en la página oficial de UCI. Permite compartir y adaptar con atribución adecuada.

## 6. Descarga reproducible

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / 'src'))

from inf8239_u01.data import download_dataset, extract_zip
from inf8239_u01.utils import DATA_RAW_DIR

BANK_URL = 'https://archive.ics.uci.edu/static/public/222/bank+marketing.zip'
csv_path = DATA_RAW_DIR / 'bank.csv'

if not csv_path.exists():
    zip_path = download_dataset(BANK_URL, 'bank-marketing.zip')
    extract_zip(zip_path)
    print('Descargado y extraído:', zip_path)
else:
    print('Ya existe:', csv_path)

Ya existe: C:\Users\reyti\AppData\Local\Temp\opencode\skillups-academy\INF8239_U01\data\raw\bank.csv


## 7. Carga

In [2]:
import pandas as pd

df = pd.read_csv(csv_path, sep=';')
print('Archivo de modelado: bank.csv (muestra aleatoria del 10 %)')
print('Forma:', df.shape)

Archivo de modelado: bank.csv (muestra aleatoria del 10 %)
Forma: (4521, 17)


## 8. Estructura

In [3]:
print('Filas:', len(df))
print('Columnas:', df.shape[1])
print('\nTipos:')
print(df.dtypes)
print('\nDuplicados exactos:', df.duplicated().sum())

Filas: 4521
Columnas: 17

Tipos:
age           int64
job          object
marital      object
education    object
default      object
balance       int64
housing      object
loan         object
contact      object
day           int64
month        object
duration      int64
campaign      int64
pdays         int64
previous      int64
poutcome     object
y            object
dtype: object

Duplicados exactos: 0


## 9. Auditoría

In [4]:
auditoria = {
    'filas': len(df),
    'columnas': df.shape[1],
    'missing_total': int(df.isna().sum().sum()),
    'missing_pct': round(df.isna().sum().sum() / df.size * 100, 4),
    'duplicados': int(df.duplicated().sum()),
    'numericas': df.select_dtypes(include='number').shape[1],
    'categoricas': df.select_dtypes(exclude='number').shape[1],
    'constantes': [c for c in df.columns if df[c].nunique(dropna=False) == 1],
    'alta_cardinalidad': [
        c for c in df.select_dtypes(exclude='number').columns
        if df[c].nunique() > 20
    ],
}
for k, v in auditoria.items():
    print(f'{k}: {v}')

filas: 4521
columnas: 17
missing_total: 0
missing_pct: 0.0
duplicados: 0
numericas: 7
categoricas: 10
constantes: []
alta_cardinalidad: []


## 10. Diccionario de datos (resumen)

El diccionario completo está en `docs/diccionario_datos.md`. Variables predictoras: `age, job, marital, education, default, balance, housing, loan, contact, day, month, campaign, pdays, previous, poutcome`. Target: `y`. Excluida por leakage: `duration`. Identificadores: no existen (no hay ID de cliente).

## 11. Target

In [5]:
print('Distribución del target:')
print(df['y'].value_counts())
print('\nPorcentaje:')
print((df['y'].value_counts(normalize=True) * 100).round(2))
print('\nNúmero de clases:', df['y'].nunique())

Distribución del target:
y
no     4000
yes     521
Name: count, dtype: int64

Porcentaje:
y
no     88.48
yes    11.52
Name: proportion, dtype: float64

Número de clases: 2


## 12. Leakage

Investigación explícita:

- **Información posterior al evento:** `duration` (duración de la última llamada) solo se conoce después de realizar el contacto y, según UCI, *highly affects the output target*. **Se excluye**.
- **Variables derivadas del target:** no se detectan.
- **Identificadores:** no existe columna de ID de cliente.
- **Variables de resultado:** únicamente `y`, que es el target legítimo.

`campaign`, `pdays`, `previous` y `poutcome` se refieren a contactos **previos o de la campaña actual** y están disponibles antes de la decisión final del cliente; se conservan como predictoras legítimas.

In [6]:
LEAKAGE_VARS = ['duration']
X = df.drop(columns=['y'] + LEAKAGE_VARS)
y = df['y']
print('Predictoras (sin leakage):', list(X.columns))

Predictoras (sin leakage): ['age', 'job', 'marital', 'education', 'default', 'balance', 'housing', 'loan', 'contact', 'day', 'month', 'campaign', 'pdays', 'previous', 'poutcome']


## 13. Train/test (mismo split para baseline y SVM)

In [7]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print('Train:', X_train.shape, '| Test:', X_test.shape)
print('Target train:', y_train.value_counts().to_dict())
print('Target test:', y_test.value_counts().to_dict())

Train: (3616, 15) | Test: (905, 15)
Target train: {'no': 3199, 'yes': 417}
Target test: {'no': 801, 'yes': 104}


## 14. Preprocessing DENTRO del pipeline

- Numéricas: `SimpleImputer(median)` + `StandardScaler`
- Categóricas: `SimpleImputer(most_frequent)` + `OneHotEncoder(handle_unknown='ignore')`

Todo el preprocessing se aprende **solo** con los datos de entrenamiento (el `ColumnTransformer` vive dentro del pipeline); el conjunto de test permanece aislado hasta la evaluación final.

In [8]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

numericas = X.select_dtypes(include='number').columns.tolist()
categoricas = X.select_dtypes(exclude='number').columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler())]), numericas),
        ('cat', Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(handle_unknown='ignore'))]), categoricas),
    ]
)
print('Preprocessor listo. Numéricas:', len(numericas), 'Categóricas:', len(categoricas))

Preprocessor listo. Numéricas: 6 Categóricas: 9


## 15. Baseline

In [9]:
from sklearn.dummy import DummyClassifier

baseline = DummyClassifier(strategy='most_frequent')
baseline.fit(X_train, y_train)
pred_baseline = baseline.predict(X_test)
print('Baseline: predice siempre la clase mayoritaria ("no") sobre el MISMO split.')

Baseline: predice siempre la clase mayoritaria ("no") sobre el MISMO split.


## 16. SVM reproducible + GridSearchCV (solo train)

In [10]:
from inf8239_u01.models import build_svm
from sklearn.model_selection import GridSearchCV, StratifiedKFold

svm_pipeline = Pipeline(steps=[
    ('preprocess', preprocessor),
    ('svm', build_svm(C=1.0, gamma='scale')),
])

param_grid = {
    'svm__model__C': [0.1, 1, 10],
    'svm__model__gamma': ['scale', 0.01, 0.1],
}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
grid = GridSearchCV(
    estimator=svm_pipeline,
    param_grid=param_grid,
    scoring='f1_macro',
    cv=cv,
    n_jobs=-1,
)
grid.fit(X_train, y_train)
print('Mejores parámetros:', grid.best_params_)
print('Mejor F1 macro (CV):', round(grid.best_score_, 4))

Mejores parámetros: {'svm__model__C': 10, 'svm__model__gamma': 'scale'}
Mejor F1 macro (CV): 0.5991


## 17. Métricas finales sobre test

In [11]:
from sklearn.metrics import (
    accuracy_score, classification_report, f1_score,
    precision_score, recall_score, roc_auc_score,
)

best = grid.best_estimator_
pred = best.predict(X_test)
proba = best.predict_proba(X_test)[:, list(best.classes_).index('yes')]

print('Accuracy :', round(accuracy_score(y_test, pred), 4))
print('Precision:', round(precision_score(y_test, pred, pos_label='yes'), 4))
print('Recall   :', round(recall_score(y_test, pred, pos_label='yes'), 4))
print('F1       :', round(f1_score(y_test, pred, pos_label='yes'), 4))
print('F1 macro :', round(f1_score(y_test, pred, average='macro'), 4))
print('ROC-AUC  :', round(roc_auc_score((y_test == 'yes').astype(int), proba), 4))
print('\n', classification_report(y_test, pred))

Accuracy : 0.8707
Precision: 0.3898
Recall   : 0.2212
F1       : 0.2822
F1 macro : 0.6056
ROC-AUC  : 0.666

               precision    recall  f1-score   support

          no       0.90      0.96      0.93       801
         yes       0.39      0.22      0.28       104

    accuracy                           0.87       905
   macro avg       0.65      0.59      0.61       905
weighted avg       0.85      0.87      0.85       905



## 18. Análisis de errores

In [12]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, pred, labels=['no', 'yes'])
tn, fp, fn, tp = cm.ravel()
print('Matriz de confusión [[TN, FP], [FN, TP]]:')
print(cm)
print('Falsos positivos (predijo yes, era no):', fp,
      '(' + str(round(fp / (fp + tn) * 100, 1)) + '% de los realmente no)')
print('Falsos negativos (predijo no, era yes):', fn,
      '(' + str(round(fn / (fn + tp) * 100, 1)) + '% de los realmente yes)')
print('El SVM pierde', fn + fp, 'de', len(y_test), 'casos;',
      'el baseline perdería', fn + tp, '(todos los yes).')

Matriz de confusión [[TN, FP], [FN, TP]]:
[[765  36]
 [ 81  23]]
Falsos positivos (predijo yes, era no): 36 (4.5% de los realmente no)
Falsos negativos (predijo no, era yes): 81 (77.9% de los realmente yes)
El SVM pierde 117 de 905 casos; el baseline perdería 104 (todos los yes).


## 19. Limitaciones

- El modelado usa `bank.csv` (10 % aleatorio) por viabilidad computacional de SVM, tal como recomienda UCI; la auditoría documenta la población completa (`bank-full.csv`).
- El target está desbalanceado (~11.5 % `yes`): las métricas macro y el recall son los indicadores más relevantes.
- Los datos provienen de una sola entidad bancaria portuguesa (2008–2010): la generalización a otros mercados o épocas no está garantizada.
- SVM con kernel RBF no produce coeficientes interpretables; no se infiere causalidad.

## 20. Conclusión

La conclusión completa (300–500 palabras) con los resultados reales se redacta en `docs/conclusiones_ejercicio01.md` una vez ejecutado el notebook; los valores numéricos provienen exclusivamente de esta ejecución y se exportan a `reports/svm_metrics.csv`, `reports/svm_cv_results.csv`, `reports/confusion_matrix.png` y `reports/svm_best.joblib` mediante `scripts/ejecutar_ejercicio01.py`.

**Estado del dataset: PENDIENTE DE APROBACIÓN DEL DOCENTE.**